# All-dataset metrics: simplified boxplots

This notebook compares **one predefined Syndri fixed-removal method** with raw expression and SPLIT. The default is Syndri removal targeting 50% of transcripts. Change `SYNDRI_TARGET_FRACTION` in the configuration cell to select another precomputed fraction.

Every plotted observation represents one tissue slice. Metrics with multiple cell-type comparisons are averaged within a slice before boxplots are calculated, preventing slices with more annotated subtypes from receiving more weight.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

ROOT = Path("/work/magroup/skrieger/Syndri/data/datasets")

DATASET_TITLES = {
    "agingbrain_merfish": "Aging brain MERFISH",
    "brain_merfish": "Brain MERFISH",
    "nsclc_cosmx": "NSCLC CosMx",
    "breast_xenium": "Breast Xenium",
}
TCELL_DATASETS = {
    "breast_xenium": "Breast Xenium",
    "nsclc_cosmx": "NSCLC CosMx",
}

# Select exactly one of the fixed-removal Syndri conditions.
SYNDRI_TARGET_FRACTION = 0.50
SYNDRI_LABEL = f"Syndri {SYNDRI_TARGET_FRACTION:.0%}"
METHOD_ORDER = [SYNDRI_LABEL, "SPLIT"]
COUNT_METHOD_ORDER = ["Raw", SYNDRI_LABEL, "SPLIT"]

COLORS = {
    "Raw": "#777777",
    SYNDRI_LABEL: "#0072B2",
    "SPLIT": "#6A3D9A",
}

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "axes.spines.top": False,
    "axes.spines.right": False,
    "figure.dpi": 130,
})

print(f"Selected Syndri method: {SYNDRI_LABEL}")


Selected Syndri method: Syndri 50%


In [ ]:
def load_per_slice_csv(dataset, family, filename, usecols=None):
    """Load one result file per slice and attach dataset/slice provenance."""
    frames = []
    root = ROOT / dataset / "results" / "metrics" / family
    for path in sorted(root.glob(f"*/{filename}")):
        if path.stat().st_size == 0:
            continue
        try:
            frame = pd.read_csv(path, usecols=usecols, low_memory=False)
        except pd.errors.EmptyDataError:
            continue
        frame["dataset"] = dataset
        frame["slice"] = path.parent.name
        frames.append(frame)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


def valid_rows(frame):
    result = frame.copy()
    if "error" in result:
        result = result.loc[result["error"].fillna("").astype(str).eq("")].copy()
    return result


def fixed_syndri_mask(frame, condition_col):
    fraction = pd.to_numeric(frame.get("target_removed_fraction"), errors="coerce")
    return (
        frame[condition_col].astype(str).str.startswith(
            "cleaned_admixed_removed_frac_", na=False
        )
        & np.isclose(fraction, SYNDRI_TARGET_FRACTION, rtol=0, atol=1e-8)
    )


def boxplot_with_points(ax, data, x, y, order, palette, ylabel=""):
    sns.boxplot(
        data=data, x=x, y=y, hue=x, order=order, hue_order=order,
        palette=palette, legend=False, width=0.58, showfliers=False,
        linewidth=1.1, ax=ax,
    )
    sns.stripplot(
        data=data, x=x, y=y, order=order, color="#222222",
        alpha=0.55, size=3.5, jitter=0.13, ax=ax,
    )
    ax.axhline(0, color="#666666", linestyle="--", linewidth=1)
    ax.set_xlabel("")
    ax.set_ylabel(ylabel)
    ax.grid(axis="y", alpha=0.22)


def add_n_labels(ax, data, category_col, order):
    ymax = ax.get_ylim()[1]
    ymin = ax.get_ylim()[0]
    y = ymax - 0.04 * (ymax - ymin)
    for position, category in enumerate(order):
        n = data.loc[data[category_col].eq(category), "slice"].nunique()
        ax.text(position, y, f"n={n}", ha="center", va="top", fontsize=8)


## Non-T-cell clustering metrics

Values are paired within slice and oriented so positive always means improvement over raw expression.


In [ ]:
non_tcell_frames = []
for dataset in DATASET_TITLES:
    frame = load_per_slice_csv(dataset, "non_tcell", "non_tcell_metrics.csv")
    if not frame.empty:
        non_tcell_frames.append(frame)

if not non_tcell_frames:
    raise FileNotFoundError("No non_tcell_metrics.csv files were found.")

non_tcell = valid_rows(pd.concat(non_tcell_frames, ignore_index=True))
non_tcell["target_removed_fraction"] = pd.to_numeric(
    non_tcell.get("target_removed_fraction"), errors="coerce"
)
non_tcell["method"] = np.select(
    [
        non_tcell["case"].eq("admixed"),
        fixed_syndri_mask(non_tcell, "case"),
        non_tcell["case"].eq("SPLIT"),
    ],
    ["Raw", SYNDRI_LABEL, "SPLIT"],
    default="",
)
non_tcell = non_tcell.loc[non_tcell["method"].ne("")].copy()

NON_TCELL_METRICS = {
    "silhouette_score": ("Silhouette", 1),
    "calinski_harabasz": ("Calinski–Harabasz", 1),
    "davies_bouldin": ("Davies–Bouldin", -1),
    "leiden_nmi_vs_label": ("Leiden NMI", 1),
    "leiden_adjusted_rand_vs_label": ("Leiden ARI", 1),
}

rows = []
for metric, (label, direction) in NON_TCELL_METRICS.items():
    values = non_tcell.pivot_table(
        index=["dataset", "slice"], columns="method", values=metric, aggfunc="mean"
    )
    if "Raw" not in values:
        continue
    for method in METHOD_ORDER:
        if method not in values:
            continue
        improvement = (values[method] - values["Raw"]) * direction
        rows.append(improvement.rename("improvement").reset_index().assign(
            method=method, metric=label
        ))

non_tcell_plot = pd.concat(rows, ignore_index=True).dropna(subset=["improvement"])
print(non_tcell_plot.groupby(["dataset", "method"])["slice"].nunique())


In [ ]:
metric_order = [label for label, _ in NON_TCELL_METRICS.values()]
fig, axes = plt.subplots(
    len(metric_order), len(DATASET_TITLES),
    figsize=(4 * len(DATASET_TITLES), 2.6 * len(metric_order)),
    sharex=True, squeeze=False,
)

for col, (dataset, title) in enumerate(DATASET_TITLES.items()):
    for row, metric in enumerate(metric_order):
        ax = axes[row, col]
        subset = non_tcell_plot.loc[
            non_tcell_plot["dataset"].eq(dataset)
            & non_tcell_plot["metric"].eq(metric)
        ]
        boxplot_with_points(
            ax, subset, "method", "improvement", METHOD_ORDER, COLORS,
            ylabel=f"{metric}\nimprovement" if col == 0 else "",
        )
        add_n_labels(ax, subset, "method", METHOD_ORDER)
        if row == 0:
            ax.set_title(title, fontweight="bold")
        if row < len(metric_order) - 1:
            ax.tick_params(axis="x", labelbottom=False)
        else:
            ax.tick_params(axis="x", rotation=25)

fig.suptitle("Cell-type separation metrics: improvement over raw", fontsize=16, y=1.005)
fig.tight_layout()
plt.show()


## T-cell metrics

Each observation is the mean paired improvement across eligible T-cell subtype comparisons within one tissue slice.


In [ ]:
def load_tcell(filename):
    frames = []
    for dataset in TCELL_DATASETS:
        frame = load_per_slice_csv(dataset, "tcell", filename)
        if not frame.empty:
            frames.append(frame)
    return valid_rows(pd.concat(frames, ignore_index=True)) if frames else pd.DataFrame()


def paired_tcell_improvement(frame, value_col, pair_cols, direction):
    data = frame.copy()
    data["target_removed_fraction"] = pd.to_numeric(
        data.get("target_removed_fraction"), errors="coerce"
    )
    data[value_col] = pd.to_numeric(data[value_col], errors="coerce")
    data = data.loc[np.isfinite(data[value_col])].copy()
    data["method"] = np.select(
        [
            data["condition"].eq("raw_X"),
            fixed_syndri_mask(data, "condition"),
            data["condition"].eq("SPLIT"),
        ],
        ["Raw", SYNDRI_LABEL, "SPLIT"],
        default="",
    )
    data = data.loc[data["method"].ne("")]
    values = data.pivot_table(
        index=["dataset", "slice", *pair_cols],
        columns="method", values=value_col, aggfunc="mean",
    )
    if "Raw" not in values:
        return pd.DataFrame()
    results = []
    for method in METHOD_ORDER:
        if method not in values:
            continue
        delta = ((values[method] - values["Raw"]) * direction).rename("improvement")
        results.append(delta.reset_index().assign(method=method))
    return pd.concat(results, ignore_index=True) if results else pd.DataFrame()

cosine = load_tcell("tcell_cosine_similarity.csv")
contamination = load_tcell("tcell_contamination_metrics.csv")

cosine_plot = paired_tcell_improvement(
    cosine, "cosine_similarity",
    ["spatial_cell_type", "reference_cell_type"], direction=1,
)
cosine_plot["metric"] = "Reference cosine similarity"

contamination_plot = paired_tcell_improvement(
    contamination, "marker_overlap_frac",
    ["target_cell_type_i", "contaminating_cell_type_j"], direction=-1,
)
contamination_plot["metric"] = "Reduced marker contamination"

tcell_plot = pd.concat([cosine_plot, contamination_plot], ignore_index=True)
tcell_slice_plot = (
    tcell_plot.groupby(["dataset", "slice", "method", "metric"], observed=True)
    ["improvement"].mean().reset_index()
)
print(tcell_slice_plot.groupby(["dataset", "method"])["slice"].nunique())


In [ ]:
tcell_metric_order = ["Reference cosine similarity", "Reduced marker contamination"]
fig, axes = plt.subplots(
    len(tcell_metric_order), len(TCELL_DATASETS),
    figsize=(10, 7), sharex=True, squeeze=False,
)
for col, (dataset, title) in enumerate(TCELL_DATASETS.items()):
    for row, metric in enumerate(tcell_metric_order):
        ax = axes[row, col]
        subset = tcell_slice_plot.loc[
            tcell_slice_plot["dataset"].eq(dataset)
            & tcell_slice_plot["metric"].eq(metric)
        ]
        boxplot_with_points(
            ax, subset, "method", "improvement", METHOD_ORDER, COLORS,
            ylabel=f"{metric}\nimprovement" if col == 0 else "",
        )
        add_n_labels(ax, subset, "method", METHOD_ORDER)
        if row == 0:
            ax.set_title(title, fontweight="bold")
        if row == len(tcell_metric_order) - 1:
            ax.tick_params(axis="x", rotation=20)
        else:
            ax.tick_params(axis="x", labelbottom=False)

fig.suptitle("T-cell metrics: improvement over raw", fontsize=16, y=1.01)
fig.tight_layout()
plt.show()


## Expression retained after cleaning

These are absolute per-slice count summaries rather than improvements from raw.


In [ ]:
COUNT_METRICS = {
    "median_genes_per_cell": "Median detected genes per cell",
    "median_transcripts_per_cell": "Median transcripts per cell",
}
count_data = non_tcell.loc[
    non_tcell["method"].isin(COUNT_METHOD_ORDER),
    ["dataset", "slice", "method", *COUNT_METRICS],
].copy()

fig, axes = plt.subplots(
    len(COUNT_METRICS), len(DATASET_TITLES),
    figsize=(4 * len(DATASET_TITLES), 6.5), sharex=True, squeeze=False,
)
for col, (dataset, title) in enumerate(DATASET_TITLES.items()):
    for row, (metric, label) in enumerate(COUNT_METRICS.items()):
        ax = axes[row, col]
        subset = count_data.loc[count_data["dataset"].eq(dataset)].dropna(subset=[metric])
        sns.boxplot(
            data=subset, x="method", y=metric, hue="method",
            order=COUNT_METHOD_ORDER, hue_order=COUNT_METHOD_ORDER,
            palette=COLORS, legend=False, width=0.58, showfliers=False,
            linewidth=1.1, ax=ax,
        )
        sns.stripplot(
            data=subset, x="method", y=metric, order=COUNT_METHOD_ORDER,
            color="#222222", alpha=0.5, size=3, jitter=0.13, ax=ax,
        )
        ax.set_xlabel("")
        ax.set_ylabel(label if col == 0 else "")
        ax.grid(axis="y", alpha=0.22)
        add_n_labels(ax, subset, "method", COUNT_METHOD_ORDER)
        if row == 0:
            ax.set_title(title, fontweight="bold")
        if row == len(COUNT_METRICS) - 1:
            ax.tick_params(axis="x", rotation=25)
        else:
            ax.tick_params(axis="x", labelbottom=False)

fig.suptitle("Cell-level expression retained after cleaning", fontsize=16, y=1.01)
fig.tight_layout()
plt.show()


## Spatial admixture (`p_admix`)

Scores are paired by cell with raw expression, then summarized as one median improvement per slice. Positive values indicate a reduction in admixture score.


In [ ]:
p_admix_rows = []
needed_columns = [
    "source_file", "slice", "cell", "cell_type", "setting", "scores",
    "removed_frac_target",
]

for dataset in DATASET_TITLES:
    root = ROOT / dataset / "results" / "metrics" / "admixture_scores"
    for path in sorted(root.glob("*/admixture_scores.csv")):
        try:
            frame = pd.read_csv(path, usecols=lambda c: c in needed_columns, low_memory=False)
        except pd.errors.EmptyDataError:
            continue
        if "slice" not in frame:
            frame["slice"] = path.parent.name
        frame["dataset"] = dataset
        frame["scores"] = pd.to_numeric(frame["scores"], errors="coerce")
        frame["removed_frac_target"] = pd.to_numeric(
            frame.get("removed_frac_target"), errors="coerce"
        )
        raw = frame.loc[frame["setting"].eq("raw"), ["cell", "cell_type", "scores"]].rename(
            columns={"scores": "raw_score"}
        )
        for method, mask in {
            SYNDRI_LABEL: (
                frame["setting"].astype(str).str.startswith("syndri_remove_", na=False)
                & np.isclose(frame["removed_frac_target"], SYNDRI_TARGET_FRACTION, rtol=0, atol=1e-8)
            ),
            "SPLIT": frame["setting"].eq("SPLIT"),
        }.items():
            cleaned = frame.loc[mask, ["cell", "cell_type", "scores"]]
            paired = cleaned.merge(raw, on=["cell", "cell_type"], how="inner", validate="many_to_one")
            if paired.empty:
                continue
            p_admix_rows.append({
                "dataset": dataset,
                "slice": path.parent.name,
                "method": method,
                "improvement": (paired["raw_score"] - paired["scores"]).median(),
                "n_cells": paired["cell"].nunique(),
            })

p_admix_plot = pd.DataFrame(p_admix_rows)
print(p_admix_plot.groupby(["dataset", "method"])["slice"].nunique())


In [ ]:
fig, axes = plt.subplots(1, len(DATASET_TITLES), figsize=(15, 4.5), sharey=True)
for ax, (dataset, title) in zip(np.atleast_1d(axes), DATASET_TITLES.items()):
    subset = p_admix_plot.loc[p_admix_plot["dataset"].eq(dataset)]
    boxplot_with_points(
        ax, subset, "method", "improvement", METHOD_ORDER, COLORS,
        ylabel="Reduction in median admixture score" if ax is np.atleast_1d(axes)[0] else "",
    )
    add_n_labels(ax, subset, "method", METHOD_ORDER)
    ax.set_title(title, fontweight="bold")
    ax.tick_params(axis="x", rotation=25)

fig.suptitle("Spatial admixture scores: improvement over raw", fontsize=16, y=1.02)
fig.tight_layout()
plt.show()


## Optional export

Uncomment and adapt `fig.savefig(...)` in any plotting cell to export publication-resolution PNG or PDF files. The tables `non_tcell_plot`, `tcell_slice_plot`, `count_data`, and `p_admix_plot` contain the exact slice-level values shown in the figures.
